# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question or material  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display


In [ ]:
# constants

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found - add OPENAI_API_KEY to .env (OpenRouter).")
elif not api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start with AIz.")
else:
    print("API key found and looks good so far.")


In [ ]:
# Shared with gradio_app.py — clients, prompts, summarize_book
from book_summarize import (
    MODEL_GPT,
    MODEL_LLAMA,
    SYSTEM_PROMPT,
    build_user_prompt,
    ollama,
    openai_client,
    summarize_book,
)

In [ ]:
# Book lookup (shared with gradio_app.py)
from book_lookup import lookup_book_google

In [ ]:
# Live streaming in Jupyter (optional): same models as book_summarize, with inline markdown updates
def llm_with_gpt(user_prompt: str, stream=False) -> str:
    response = openai_client.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        stream=stream,
    )
    if stream:
        text = ""
        display_handle = display(Markdown(""), display_id=True)
        for chunk in response:
            text += chunk.choices[0].delta.content or ""
            update_display(Markdown(text), display_id=display_handle.display_id)
        return text
    return response.choices[0].message.content

In [ ]:
#llm call with llama

def llm_with_llama(user_prompt: str, stream=False) -> str:
    response = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
            ],
        stream=stream
    )

    if stream:
        text = ""
        display_handle = display(Markdown(""), display_id=True)
        for chunk in response:
            text += chunk.choices[0].delta.content or ''
            update_display(Markdown(text), display_id=display_handle.display_id)
        return text
    return response.choices[0].message.content

In [ ]:
# summarize_book is imported from book_summarize (cell 3), same as gradio_app.py

In [ ]:
# lookup book with google books api
meta = lookup_book_google("Atomic Habits", "James Clear")
print(meta)

In [ ]:
# Get gpt-4o-mini to answer
summary = summarize_book(meta, llm_with_gpt)
print(summary)

In [ ]:
# Get Llama 3.2 to answer
summary = summarize_book(meta, llm_with_llama)
print(summary)

In [ ]:
#llm call with llama with stream
summary = summarize_book(meta, llm_with_llama,True)
# print(summary)

In [ ]:
#llm call with gpt with stream
summary = summarize_book(meta, llm_with_gpt,True)